# 3D reconstruction via colmap and nerfstudio splatfacto

* this currently only works in linux. expects a conda environment setup. tested with ubuntu 24.04.3 LTS

## setup

In [ ]:
import os
import sys
from pathlib import Path

In [ ]:
# Locate dt4ag_config.py (it lives in the pipeline/ directory, one level above
# notebooks/) and import the config loader. Walking upward keeps this working
# whether the notebook runs from the repo checkout or from a copy sitting next
# to the data.
def _find_pipeline_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "dt4ag_config.py").is_file():
            return candidate
    raise FileNotFoundError(
        f"could not find dt4ag_config.py by walking up from {here}. Run this "
        "notebook from inside the jsps-dt4ag repo, or add the directory "
        "containing dt4ag_config.py to PYTHONPATH."
    )


pipeline_root = _find_pipeline_root()
if str(pipeline_root) not in sys.path:
    sys.path.insert(0, str(pipeline_root))

from dt4ag_config import ConfigError, find_config, load_config

print("pipeline_root:", pipeline_root)

In [ ]:
print(os.getcwd())

In [ ]:
#ensure colmap is installed and check version
!colmap

In [ ]:
# Load the run configuration.
#
# Everything that used to be typed into these cells by hand (data root, dataset
# selection, colmap flags, training and export settings) now comes from an INI
# file. See configs/README.md for what every key does.
#
# Precedence: the DT4AG_CONFIG environment variable, else configs/example.ini
# found by walking up from pipeline_root. To pick a config from here:
#     os.environ["DT4AG_CONFIG"] = "/path/to/my-run.ini"
config_path = find_config(start=pipeline_root)
cfg = load_config(config_path)
print(cfg.describe())

In [ ]:
# Run identity.
#
# run_date, todays_run_count and colmap_ver used to be typed by hand every run.
# All three are now derived: the date from today's clock, the count by
# inspecting the run directories already present for this dataset, the COLMAP
# version by invoking `colmap`. Any of them can still be pinned in [run] to
# reproduce an old run id.
project_id = cfg.make_run_id()
print('project_id:', project_id)

# One row per run id issued. Re-running this cell appends another row.
print('run log:', cfg.append_run_log(project_id))

# Root paths, all hanging off [paths] data_root.
base_dir = cfg.data_root
colmap_dir = cfg.colmap_dir       # the colmap workspace
dataset_dir = cfg.datasets_dir    # the dataset (2d raw images) dir
export_dir = cfg.exports_dir      # nerfstudio exports (like .ply splats)
output_dir = cfg.outputs_dir      # nerfstudio outputs (like config.yml)
export_3dgs = cfg.export_3dgs

In [ ]:
# Use dictionary to store path objects
paths_dict = {
    "data_root": base_dir,
    "colmap_dir": colmap_dir,
    "dataset_dir": dataset_dir,
    "export_dir": export_dir,
    "output_dir": output_dir,
}

In [ ]:
for key, value in paths_dict.items():
    print(f"Key: {key}, Value: {value}", "Type:", type(value))


## colmap related commands

In [ ]:
# list dataset contents via bash
!ls -ll $dataset_dir

In [ ]:
# list dataset contents via python
contents = list(Path(dataset_dir).iterdir())
for item in contents: print(item)

In [ ]:
# Resolve the dataset image directory.
#
# This used to be four chained loops doing `if str(subdir_id) in str(path)`.
# They matched on substrings and never broke out of the loop, so when two
# sibling directories both matched, the LAST one silently won. The path is now
# stated explicitly as [dataset] images_subpath and its existence is checked
# when the config loads.
colmap_reconstruction_images_path = cfg.images_path

print('images_subpath (relative to datasets dir):', cfg.images_rel)
print('colmap_reconstruction_images_path:', colmap_reconstruction_images_path)
print('path exists:', colmap_reconstruction_images_path.exists())

In [ ]:
# Sanity check. An empty image directory is a silent-failure trap: COLMAP will
# happily run on nothing and produce nothing, and the upstream masking script
# has a known no-op mode that leaves exactly that behind.
#
# The walk is RECURSIVE on purpose. With single_camera_per_folder the images sit
# in one subdirectory per camera rather than flat in this directory, so a
# non-recursive check reports zero images on a perfectly good dataset.
image_suffixes = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}
image_files = sorted(
    p for p in colmap_reconstruction_images_path.rglob('*')
    if p.is_file() and p.suffix.lower() in image_suffixes
)
print('image files found:', len(image_files))
if not image_files:
    raise RuntimeError(
        f'no image files in {colmap_reconstruction_images_path} (searched '
        f'recursively). Check [dataset] images_subpath in {cfg.source}'
    )
subdirs = sorted({p.parent.relative_to(colmap_reconstruction_images_path).as_posix()
                  for p in image_files})
print('camera dirs:', subdirs if subdirs != ['.'] else '(flat)')
print('first:', image_files[0].name, '| last:', image_files[-1].name)

In [ ]:
# list the chosen image directory via bash
!ls -ll $colmap_reconstruction_images_path

In [ ]:
# Path components used to name the colmap workspace, the nerfstudio output
# directory and the export file. `dataset_rel` mirrors the dataset's path
# across the colmap/ and outputs/ trees.
#
# This used to walk four `.parent` levels and so assumed a depth-4 hierarchy,
# which is why the dataset-selection cell carried an "uncomment this line if
# there is no fourth level" escape hatch. Taking the configured relative path
# directly is depth-agnostic and needs no escape hatch.
dataset_rel = cfg.images_rel
images_dir_name = colmap_reconstruction_images_path.name
images_parent_name = colmap_reconstruction_images_path.parent.name

print('dataset_rel:', dataset_rel)
print('images_dir_name:', images_dir_name)
print('images_parent_name:', images_parent_name)

In [ ]:
colmap_reconstruction_workspace_path = cfg.colmap_workspace(project_id)
print('colmap_reconstruction_workspace_path', colmap_reconstruction_workspace_path)
print('path exists:', colmap_reconstruction_workspace_path.exists())
colmap_reconstruction_workspace_path.mkdir(parents=True, exist_ok=True)
print('path exists:', colmap_reconstruction_workspace_path.exists())

### execute colmap automatic reconstruction

In [ ]:
# `auto` infers from the image directory name: a name mentioning "frames" came
# from a video, anything else is treated as individual images. Override with
# [colmap] data_type.
colmap_data_type = cfg.resolve_colmap_data_type()

print('colmap_data_type', colmap_data_type)
print('colmap_reconstruction_images_path', colmap_reconstruction_images_path)
print('colmap_reconstruction_workspace_path', colmap_reconstruction_workspace_path)

In [ ]:
colmap_cmd = ' '.join(filter(None, [
    'colmap automatic_reconstructor',
    f'--workspace_path {colmap_reconstruction_workspace_path}',
    f'--image_path {colmap_reconstruction_images_path}',
    f'--data_type {colmap_data_type}',
    f'--single_camera {cfg.colmap_single_camera}',
    f'--single_camera_per_folder {cfg.colmap_single_camera_per_folder}',
    f'--dense {cfg.colmap_dense}',
    cfg.colmap_extra_args,
]))
print(colmap_cmd)
!{colmap_cmd}
if globals().get('_exit_code', 0):
    raise RuntimeError(f'colmap exited {globals()["_exit_code"]}: {colmap_cmd}')

## run nerfstudio commands

### prep for ns-process-data

In [ ]:
scene = cfg.scene_type                       # 'images' or 'video'
ns_images_dir = colmap_reconstruction_images_path
ns_video_path = cfg.video_path

print('scene:', scene)
print('ns_images_dir:', ns_images_dir, ns_images_dir.exists())
print('ns_video_path:', ns_video_path or '(unset)')

In [ ]:
ns_colmap_dir = colmap_reconstruction_workspace_path
print('ns_colmap_dir:',ns_colmap_dir,ns_colmap_dir.exists())

In [ ]:
!ls $ns_colmap_dir

### run ns-process-data

In [ ]:
if scene == 'images':
    ns_process_cmd = ' '.join(filter(None, [
        'ns-process-data images',
        f'--data {ns_images_dir}',
        f'--output-dir {ns_colmap_dir}',
        '--skip-colmap' if cfg.skip_colmap else '',
        f'--colmap-model-path {cfg.colmap_model_path}',
    ]))
elif scene == 'video':
    # Not currently exercised by this pipeline.
    ns_process_cmd = (
        f'ns-process-data video --data {ns_video_path} --output-dir {ns_colmap_dir}'
    )
else:
    raise ValueError(f'unsupported [nerfstudio] scene_type: {scene!r}')

print(ns_process_cmd)
!{ns_process_cmd}
if globals().get('_exit_code', 0):
    raise RuntimeError(f'ns-process-data exited {globals()["_exit_code"]}')
print("Data Processing Succeeded!")

### prep for ns-train

In [ ]:
!ls $ns_colmap_dir

In [ ]:
# prep output dir
ns_output_dir = cfg.output_parent
print('ns_output_dir:', ns_output_dir, ns_output_dir.exists())

### run ns-train

In [ ]:
num_steps = cfg.max_num_iterations   # [train] max_num_iterations, splatfacto default is 30000
ns_train_cmd = ' '.join([
    f'ns-train {cfg.train_method}',
    f'--data {ns_colmap_dir}',
    f'--pipeline.model.use_scale_regularization {cfg.use_scale_regularization}',
    f'--pipeline.model.background_color {cfg.background_color}',
    f'--output-dir {ns_output_dir}',
    f'--viewer.quit-on-train-completion {cfg.quit_on_train_completion}',
    f'--max-num-iterations {num_steps}',
    f'--logging.local-writer.max-log-size {cfg.max_log_size}',
])
print(ns_train_cmd)
!{ns_train_cmd}
if globals().get('_exit_code', 0):
    raise RuntimeError(f'ns-train exited {globals()["_exit_code"]}')

## export splat

### somehow get the config.yml path!!

In [ ]:
# nerfstudio names the experiment after the --data directory, which is the
# colmap workspace, which is named after the run id.
config_yml_parent_path = ns_output_dir / project_id / cfg.train_method
print('config_yml_parent_path', config_yml_parent_path, config_yml_parent_path.exists())

In [ ]:
# Pick the training run to export from.
#
# Timestamped run directories sort chronologically, so the newest is last. But
# "newest" is not the same as "usable": a crashed ns-train still leaves behind
# a directory holding config.yml and dataparser_transforms.json and no weights
# at all, and ns-export against that dies deep inside the checkpoint loader
# with an unhelpful message. So require BOTH artefacts, and report what each
# candidate actually contains rather than picking blind.
run_dirs = sorted(p for p in config_yml_parent_path.iterdir() if p.is_dir())
if not run_dirs:
    raise RuntimeError(f'no training run directories under {config_yml_parent_path}')

usable = []
for path in run_dirs:
    has_config = (path / 'config.yml').is_file()
    ckpts = sorted((path / 'nerfstudio_models').glob('*.ckpt'))
    print(f'{path.name}  config.yml={has_config}  checkpoints={len(ckpts)}'
          + (f'  latest={ckpts[-1].name}' if ckpts else ''))
    if has_config and ckpts:
        usable.append((path, ckpts[-1]))

if not usable:
    raise RuntimeError(
        f'no run directory under {config_yml_parent_path} has both a config.yml '
        f'and a checkpoint in nerfstudio_models/. Training did not get far '
        f'enough to save weights; re-run ns-train before exporting.'
    )

selected_run_dir, checkpoint_path = usable[-1]
config_yml_path = selected_run_dir / 'config.yml'

print('selected run dir  :', selected_run_dir.name)
print('config_yml_path   :', config_yml_path)
print('checkpoint_path   :', checkpoint_path)
print('checkpoint bytes  :', checkpoint_path.stat().st_size)

In [ ]:
print('dataset_rel:', dataset_rel)
print('images_dir_name:', images_dir_name, '| images_parent_name:', images_parent_name)
print('project_id:', project_id)

In [ ]:
# Export filename records what produced it: dataset, run id, platform, env,
# iteration count and colmap data type. The labels come from [export].
ns_export_dir = config_yml_path.parent / 'exports'
ns_export_filename = '_'.join([
    str(images_parent_name),
    str(project_id),
    'splat',
    cfg.platform_label,
    cfg.env_label,
    f'{num_steps}steps',
    colmap_data_type,
]) + '.ply'
print('ns_export_dir:', ns_export_dir)
print('ns_export_filename:', ns_export_filename)

In [ ]:
ns_export_fullpath = ns_export_dir / ns_export_filename

print ("Export splat to:",ns_export_fullpath)
print('checkpoint path exists:',ns_export_fullpath.parent.parent.exists())

### run ns-export

In [ ]:
ns_export_cmd = ' '.join([
    f'ns-export {cfg.export_format}',
    f'--load-config {config_yml_path}',
    f'--output-dir {ns_export_dir}',
    f'--output-filename {ns_export_filename}',
])
print(ns_export_cmd)
!{ns_export_cmd}
if globals().get('_exit_code', 0):
    raise RuntimeError(f'ns-export exited {globals()["_exit_code"]}')

# A zero exit status from ns-export is not proof that a splat landed on disk.
# Check the file, and check it is not a bare PLY header with no vertices.
if not ns_export_fullpath.is_file():
    raise RuntimeError(
        f'ns-export reported success but {ns_export_fullpath} does not exist. '
        f'Directory contains: '
        f'{sorted(p.name for p in ns_export_dir.iterdir()) if ns_export_dir.is_dir() else "(no such directory)"}'
    )
export_bytes = ns_export_fullpath.stat().st_size
if export_bytes < 1024:
    raise RuntimeError(
        f'{ns_export_fullpath} is only {export_bytes} bytes, which is a header '
        f'and no geometry. Training almost certainly produced nothing usable.'
    )
print('Export successful!')
print('exported file :', ns_export_fullpath)
print('bytes         :', export_bytes, f'({export_bytes / 1024 / 1024:.2f} MiB)')

### export 3dgs to point cloud (note this increases file size like 1000X!!)

In [ ]:
# Optional: convert the exported 3DGS splat to a point cloud.
# input path example:     <data_root>/outputs/<dataset_rel>/<run_id>/<method>/<timestamp>/exports/<name>.ply
# transform path example: <data_root>/colmap/<dataset_rel>/<run_id>/transforms.json
a_3dgs_input_path = ns_export_fullpath
a_3dgs_transforms_path = ns_colmap_dir
a_3dgs_output_path = str(a_3dgs_input_path.parent / a_3dgs_input_path.stem) + '_3dgs-to-pc.ply'

print(a_3dgs_input_path)
print(a_3dgs_transforms_path)
print(a_3dgs_output_path)

In [ ]:
if export_3dgs:
    gauss_to_pc = os.path.expanduser(cfg.gauss_to_pc_script)
    gauss_to_pc_cmd = ' '.join([
        f'python {gauss_to_pc}',
        f'--input_path {a_3dgs_input_path}',
        f'--transform_path {a_3dgs_transforms_path}',
        f'--output_path {a_3dgs_output_path}',
    ])
    print(gauss_to_pc_cmd)
    !{gauss_to_pc_cmd}
    if globals().get('_exit_code', 0):
        raise RuntimeError(f'gauss_to_pc exited {globals()["_exit_code"]}')